In [2]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

In [3]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
if google_api_key:
    print(f"Google API Key is set. and begins with {google_api_key[:4]}...")
else:    print("Google API Key is not set. Please set the GOOGLE_API_KEY environment variable.")

Google API Key is set. and begins with AIza...


In [4]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [19]:
system_message =  "You are a Helpful Assistant."

## New Callback

We need to write a function called :

chat(message, history)

In [20]:
def chat(message, history):
    return "bananas"

In [24]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [25]:
def chat(message, history):
    return f"You said '{message}' and the history is {history} but I still say bananas"

In [26]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [29]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)
    return response.choices[0].message.content

In [30]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [5]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    stream = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content
        yield response

In [32]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


#One short prompting

In [ ]:
system_message = "You are a helpful assistant in a restaurant. You should try to gently encourage \
the customer to try items that are on menu for that special day. Platters + drinks with soup are 30% off, and most other items are 20% off. \
For example, if the customer says 'I .am so hungry. What is good for me today and what should i try?', \
you could reply something like, 'We have some delicious platters and drinks with soup that are 30% off today! " \
"I highly recommend trying our special of the day, which is a platter of grilled chicken with a side of soup. " \
"It's a great deal and very tasty!'"

In [6]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
system_message += "\nIf the customer asks for other items like burger, sandwiches, you should respond that they are not included in menu due high customer flow. \
but remind the customer to look at special items!"

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'alcohol' in message.lower():
        relevant_system_message += " Restaurant does not sell alcohol beverages; if you are asked for alcohol beverages, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0 ].delta.content or ''
        yield response

In [10]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
